# BUSI 半监督分割实验 Notebook（注释版）

这个 notebook 主要记录了你当前这套 BUSI clean baseline 工程在 AutoDL 上的运行流程，包含：
- 代码解压与环境安装
- BUSI 数据预处理与 manifest 生成
- 8:1:1 数据划分与 1/2、1/4、1/8 labeled 子集构建
- 纯监督工程自检（overfit_8_samples）
- 全监督训练（sup_full）
- 仓库打包

为了方便后续回看，我在每个代码块前都补了一段说明，解释这格代码的作用、输入输出和它在整体流程中的位置。


## 代码块 1：解压实验工程代码

这一格的作用是把你已经打包好的实验仓库解压到 AutoDL 的工作目录中。

这里默认压缩包文件是 `autodl-fs/busi_ssl_baseline_s5-1.zip`，解压目标是 `/root/autodl-fs/`。  
解压后，后续所有脚本、配置文件、训练入口都会从这个目录读取。

你后续如果换了新的仓库压缩包，这一格通常只需要改两个地方：
- 压缩包文件名
- 解压目标目录

执行前建议确认：
- 压缩包确实存在
- 目标目录有写权限



In [1]:
!unzip -q autodl-fs/busi_ssl_baseline_s5-1.zip -d /root/autodl-fs/

## 代码块 2：检查 Python 环境并安装依赖

这一格做两件事：

1. `python -V`：确认当前运行环境的 Python 版本  
2. `pip install -r .../requirements.txt`：安装项目运行所需依赖

这一步的目的不是训练，而是确保工程依赖齐全，后面运行数据处理、训练脚本时不会因为缺包报错。

通常在以下情况需要重新执行这一格：
- 新开一个 AutoDL 环境
- 换了基础镜像
- 更新了仓库中的 `requirements.txt`

如果这里安装失败，后面的代码块一般都不要继续跑。



In [ ]:
!python -V
!pip install -r autodl-fs/busi_ssl_baseline/busi_ssl_baseline/requirements.txt

## 代码块 3：生成 BUSI 的 manifest 和离线 merged mask

这一格运行 `prepare_busi_manifest.py`，是整个实验流程里非常关键的数据准备步骤。

它会做这些事：
- 扫描 BUSI 原始数据目录
- 默认排除 `normal` 类
- 只保留 `benign` 和 `malignant`
- 查找一张图像对应的一个或多个 mask
- 将多个 mask 逻辑 OR 合并成一张二值 `merged mask`
- 生成 `manifest.csv`
- 将合并后的 mask 保存到 `data_meta/merged_masks/`

参数说明：
- `--dataset-root`：BUSI 原始数据根目录
- `--output-root`：生成的元数据目录
- `--overwrite-merged-masks`：如果已存在旧 mask，则覆盖重建

这一步跑完后，后续 dataset、split、训练都会依赖这里生成的结果。



In [4]:
!python autodl-fs/busi_ssl_baseline/busi_ssl_baseline/tools/prepare_busi_manifest.py \
  --dataset-root /root/autodl-fs/Dataset_BUSI_with_GT \
  --output-root autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta \
  --overwrite-merged-masks

## 代码块 4：检查 manifest 和 merged mask 是否生成成功

这一格不是训练步骤，而是数据准备后的快速人工检查。

它依次查看：
- `data_meta` 目录里是否生成了目标文件
- `merged_masks` 目录里是否已经有合并后的 mask
- `manifest.csv` 的前几行内容

通过这一步，你可以快速确认：
- `prepare_busi_manifest.py` 是否真的执行成功
- 生成的文件是否落在了预期位置
- `manifest.csv` 字段是否看起来合理

如果这里输出异常，不建议直接进入 split 或训练阶段。



In [ ]:
!ls autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta
!ls autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/merged_masks | head
!head autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/manifest.csv

## 代码块 5：生成 8:1:1 划分 + 1/2 labeled 的 split

这一格运行 `make_splits.py`，生成第一套实验划分：
- train / val / test = 8 : 1 : 1
- train 内部的 labeled 比例 = 1/2

也就是说，这套 split 主要用于：
- 1/2 labeled 的纯监督实验
- 1/2 labeled 的半监督实验

关键参数：
- `--manifest-path`：上一步生成的 `manifest.csv`
- `--output-dir`：这套划分写到哪个目录
- `--seed 0`：固定随机种子，保证可复现
- `--val-fraction 0.1`
- `--test-fraction 0.1`
- `--labeled-fraction 0.5`

输出目录里通常会包含：
- `train.txt`
- `val.txt`
- `test.txt`
- `labeled_subset.txt`
- `unlabeled_pool.txt`
- 若干 verify 用的小 split



In [14]:
!python autodl-fs/busi_ssl_baseline/busi_ssl_baseline/tools/make_splits.py \
  --manifest-path autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/manifest.csv \
  --output-dir autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/splits/seed0_labeled_1of2 \
  --seed 0 \
  --val-fraction 0.1 \
  --test-fraction 0.1 \
  --labeled-fraction 0.5

## 代码块 6：生成 8:1:1 划分 + 1/4 labeled 的 split

这一格和上一格逻辑相同，只是把 train 内 labeled 比例改成了 `1/4`。

用途：
- 构造 1/4 标注量场景下的纯监督与半监督实验

这里最关键的变化是：
- `--labeled-fraction 0.25`

其余参数保持一致，这样可以保证：
- train / val / test 不变
- 变化的只是 train 内部 labeled 和 unlabeled 的拆分方式



In [16]:
!python autodl-fs/busi_ssl_baseline/busi_ssl_baseline/tools/make_splits.py \
  --manifest-path autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/manifest.csv \
  --output-dir autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/splits/seed0_labeled_1of4 \
  --seed 0 \
  --val-fraction 0.1 \
  --test-fraction 0.1 \
  --labeled-fraction 0.25

## 代码块 7：生成 8:1:1 划分 + 1/8 labeled 的 split

这一格用于构造低标注场景：
- train / val / test = 8 : 1 : 1
- train 内 labeled = 1/8

用途：
- 这是最接近后续半监督研究重点的低标注设置
- 常常也是最值得重点比较的设置

关键参数：
- `--labeled-fraction 0.125`



In [18]:
!python autodl-fs/busi_ssl_baseline/busi_ssl_baseline/tools/make_splits.py \
  --manifest-path autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/manifest.csv \
  --output-dir autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/splits/seed0_labeled_1of8 \
  --seed 0 \
  --val-fraction 0.1 \
  --test-fraction 0.1 \
  --labeled-fraction 0.125

## 代码块 8：检查三套 split 的样本数量是否符合预期

这一格用 `wc -l` 统计三套 split 目录中各个 txt 文件的行数。

主要检查：
- `train.txt / val.txt / test.txt` 是否一致
- `labeled_subset.txt` 是否分别约为 train 的 1/2、1/4、1/8
- `unlabeled_pool.txt + labeled_subset.txt` 是否等于 train.txt

这是一个非常重要的口径检查步骤。  
如果这里数量不对，后面的实验结果口径也会不对。



In [ ]:
!wc -l autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/splits/seed0_labeled_1of2/*.txt
!wc -l autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/splits/seed0_labeled_1of4/*.txt
!wc -l autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/splits/seed0_labeled_1of8/*.txt

## 代码块 9：检查旧的默认 split 目录（seed0）

这一格是在看 `data_meta/splits/seed0` 这一套旧目录的文件情况。

它更像是历史检查或对比检查，而不是当前正式实验必须的一步。  
如果你后续已经明确使用：
- `seed0_labeled_1of2`
- `seed0_labeled_1of4`
- `seed0_labeled_1of8`

那么这一格更多是为了确认旧目录里有哪些文件，避免自己混淆。

如果你后续想让 notebook 更精简，这一格其实可以保留为“可选检查”。



In [ ]:
!ls autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/splits/seed0
!wc -l autodl-fs/busi_ssl_baseline/busi_ssl_baseline/data_meta/splits/seed0/*.txt

## 代码块 10：查看 `train_sup.py` 的命令行参数

这一格的作用是查看纯监督训练入口脚本支持哪些参数。

它主要帮助你确认：
- `--config` 这样的参数名是否正确
- 入口脚本是否已经从占位状态变成可运行状态
- 后续正式训练该用什么调用方式

在开始第一次正式训练前，执行这一格是有意义的。



In [ ]:
!python autodl-fs/busi_ssl_baseline/busi_ssl_baseline/train_sup.py --help

## 代码块 11：运行 overfit_8_samples 工程自检

这一格不是正式实验，而是工程链路自检。

它会使用 `configs/verify/overfit_8_samples.yaml` 运行一个非常小的纯监督任务，通常只在 8 张图像上训练和验证。  
目的不是看泛化，而是确认：

- 数据读取是否正常
- 模型能否前向与反向传播
- loss 是否能下降
- evaluator 是否能工作
- checkpoint 和日志是否会正常落盘

如果这一格失败，一般不建议直接跑正式实验。



In [ ]:
!python /root/autodl-fs/busi_ssl_baseline/busi_ssl_baseline/train_sup.py \
  --config /root/autodl-fs/busi_ssl_baseline/busi_ssl_baseline/configs/verify/overfit_8_samples.yaml

## 代码块 12：运行全监督训练（sup_full）

这一格对应正式纯监督参考实验。

它会使用 `configs/experiments/sup_full.yaml` 启动训练，特点是：
- train_split 使用 `train.txt`
- 也就是整个训练集都作为有标签样本使用
- val / test 仍然来自固定的 8:1:1 划分

这一组实验的意义是：
- 给出全监督参考上界
- 帮助判断低标注纯监督与半监督的提升空间
- 为后续引入动态学习率、SSL 机制提供参照结果

建议在这一格执行前先确认：
- 配置中的路径都正确
- `manifest.csv`、`merged_masks/`、`splits/` 都已经生成



In [ ]:
!python /root/autodl-fs/busi_ssl_baseline/busi_ssl_baseline/train_sup.py \
    --config autodl-fs/busi_ssl_baseline/busi_ssl_baseline/configs/experiments/sup_full.yaml

## 代码块 13：打包当前实验仓库

这一格的作用是把当前仓库重新压缩成 zip 包，便于：
- 备份
- 下载
- 迁移到别的环境
- 发给我继续审查

注意：
- 这一格打包的是代码仓库目录
- 一般不建议把大型训练输出、缓存文件也一起打进去
- 如果输出目录很大，打包前最好确认 `.gitignore` 或手动清理是否合理



In [26]:
!zip -rq busi_ssl_baseline.zip autodl-fs/busi_ssl_baseline